# Dunnhumby 마지막 주 seed49 — M3·M4-B 누락 비교

**L4 GPU 노트북 한 개에서 M3 → M4-B 두 모델만 순차 학습합니다. 현재 H&M 실행과 별도 런타임에서 실행하세요.**

- 학습 DAY1~704 전체 / test DAY705~711. validation·holdout 없음.
- seed49, 각300epoch 고정. 마지막 checkpoint만 test 1회 평가. 조기종료·최고점 선택 없음.
- 신규상품 추천, train pair는 정답·후보에서 제외. MIN_ITEM_INTER=1, uniform K=1.
- ID64, 2층, batch8192, lr=.0005, L2=.001. 각 모형 단일 optimizer. CLV 관측창365일.
- **M1·M5-B는 기존 마지막주 seed49 결과만 재사용**하며 추가 학습·재평가하지 않습니다. 원본 해시와 입력/설정/분할/가중치/beta/142지표/300epoch 이력을 검사합니다.
- **M3:** N/V 두 축 중심화 가치그래프 + 일반 BPR. N과 V 모두 포함합니다. 학습구간 target CV=.20으로 beta 보정.
- **M4-B:** 이진 그래프 + raw=(1+.5q_N)×(1+.5q_V×상품가격백분위×경제구간적합도). CLV·사용자경제·아이템경제 무효행 raw=1, 전체 학습행 평균 정규화. 기존 Dunnhumby M5-B와 동일한 가중치 SHA를 요구합니다. 현행 H&M의 무효입력 처리와는 차이가 있으므로 동일 구현이라고 서술하지 않습니다.

## 판독 규칙 — 새 비교의 기술적 진단
M5-B의 전체 사용자 가격·구매금액 가중 적중값@10 및 가중 NDCG@10이 각각 M1·M3·M4-B보다 높은지, 여섯 Recall/NDCG@10·20·50이 각각 M1 및 지표별 더 좋은 단독모형의99% 이상인지 **각각 보고**합니다. 기존 실험의 판정을 변경하지 않습니다.

모든142지표에 대해 (M5-B−M3)−(M4-B−M1)의 **절대 차이**를 저장합니다. 단일 시드의 기술적 비교이며 통계적 유의성·CLV 배정귀속·인과적 상호작용을 주장하지 않습니다. 이미 노출된 test이므로 독립 확증이 아니며, 같은 test에 맞춘 구조/epoch/가중치 재선택에 사용하지 않습니다. 최종10시드 평균 보고를 대체하지 않습니다.

소형 CPU 검사와 실제 기준 ZIP 검증 완료. 전체 데이터/L4 학습은 아직 실행하지 않았습니다.

## 1. L4 확인·Drive 연결·고정 코드 로드
런타임 → 런타임 유형 변경 → **L4 GPU**를 선택하고 실행하세요.

In [ ]:
from pathlib import Path
import subprocess, sys, json, torch
assert torch.cuda.is_available(), 'L4 GPU 런타임을 연결하세요.'
assert 'L4' in torch.cuda.get_device_name(0), '런타임 유형을 L4로 변경하세요.'
from google.colab import drive
drive.mount('/content/drive')
SOURCE_COMMIT = 'f3e4046f746346e35dcd2adc23ce2983b6b88862'
REPO = Path('/content/clv-dh-m3-m4b-s49-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git','clone','-q','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','-q','origin'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','-q','--detach',SOURCE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip() == SOURCE_COMMIT
sys.path.insert(0,str(REPO))
print('source:', SOURCE_COMMIT, '| GPU:', torch.cuda.get_device_name(0))


## 2. 기존 M1·M5-B 원본 확인 — 학습 없음
Drive의 정확한 기존 실행 경로를 먼저 확인합니다. 없으면 `m2_m5_lastweek_dunnhumby_seed49_results.zip` **한 개**를 업로드하세요. 다른 결과나 수정된 파일은 거절하며 기준모형을 자동 재학습하지 않습니다.

In [ ]:
import clv_dh_m3_m4b_lastweek_completion as screen
reference_path = Path(screen.REFERENCE_PATH)
if not reference_path.is_file():
    from google.colab import files
    print('기존 결과 ZIP 한 개를 선택하세요: m2_m5_lastweek_dunnhumby_seed49_results.zip')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('결과 ZIP 한 개만 업로드하세요.')
    reference_path = Path(next(iter(uploaded)))
reference = screen.load_reference(reference_path)
print('원본 SHA 확인:', screen.REFERENCE_SHA)
print('재사용:', screen.M1, '/', screen.M5_B, '| 새 학습은 M3·M4-B만')


## 3. 동일 입력·그래프·행 가중치 준비 — 학습 없음
불일치하면 여기서 중단합니다. 기존 실행 경로를 바꾸거나 검사를 우회하지 마세요.

In [ ]:
cfg = screen.configure()
assert cfg.dataset == 'dunnhumby' and cfg.seeds == (49,) and cfg.epochs == 300
assert screen.common.models_for(cfg) == (screen.M3,screen.M4_B)
prepared = screen.prepare(cfg, reference_path=reference_path)
assert set(prepared['data']['splits']) == {'test'}
assert prepared['protocol']['total_fits'] == 2
print('체크포인트/보고서 경로:', prepared['run_dir'])


## 4. M3 → M4-B 순차 학습
각300epoch, 총2회입니다. 매epoch optimizer·난수상태를 저장하며 로그는25epoch 간격입니다. 런타임 중단 후 같은 노트북을 다시 실행하면 마지막 저장 epoch에서 재개합니다. 완료 모델은 다시 학습·평가하지 않습니다.

In [ ]:
result = screen.run(cfg,prepared)
assert len(result['absolute']) == 4
print(json.dumps(result['reading'],ensure_ascii=False,indent=2))


## 5. 네 모델 전체 사용자 비교
전체·저/중/고CLV·노출 142지표의 원본은 ZIP에 포함됩니다. 아래는 주요12지표입니다. 상대변화율은%, 상호작용 표는 절대값으로 단위가 다릅니다.

In [ ]:
metrics = [f'{m}@{k}' for k in (10,20,50) for m in ('recall','ndcg','price_purchase_amount_weighted_hit','vndcg')]
labels = {screen.M1:'M1 (기존)',screen.M3:'M3 (추가)',screen.M4_B:'M4-B (추가)',screen.M5_B:'M5-B (기존)'}
view = result['absolute'].set_index('model_id').loc[list(screen.MODELS),metrics].T.rename(columns=labels)
print(view.to_string())
comparison = result['comparison'].copy()
comparison['model_id'] = comparison.model_id.map(labels)
comparison['reference'] = comparison.reference.map(labels)
print('\n상대변화율(%)')
print(comparison[comparison.metric.isin(metrics)].pivot(index='metric',columns=['model_id','reference'],values='relative_change_pct').to_string())
print('\n기술적 결합 차이 — 절대값, 유의성/인과성 주장 없음')
print(result['interaction'].set_index('metric').loc[metrics].to_string())
print('\n작동 진단')
print(result['diagnostics'].to_string(index=False))


## 6. 결과 ZIP 다운로드
`absolute.csv`, `comparison.csv`, `interaction.csv`, `diagnostics.csv`, `result.json`, `protocol.json`, `reference_audit.json`을 포함합니다. 학습이력은 result.json에, 체크포인트는 Drive에 남습니다.

In [ ]:
from zipfile import ZipFile,ZIP_DEFLATED
from google.colab import files
zip_path = Path('/content/dh_m3_m4b_lastweek_seed49_results.zip')
with ZipFile(zip_path,'w',ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path,arcname=Path(path).name)
    for name in ('protocol.json','reference_audit.json'):
        archive.write(prepared['run_dir']/name,arcname=name)
print(json.dumps(result['paths'],ensure_ascii=False,indent=2))
files.download(str(zip_path))
